# 第 7 週 Part B：診所掛號系統（個人作品）

**規則：** 個人獨立完成；可以使用任何 AI 工具；**禁止與其他同學討論**。

**流程（照順序做）：**
1. 在下一格寫好**規格書**（可以請 AI 幫忙寫初稿，但你要讀過並修改成你要的樣子）。
2. 把規格書交給 AI，請它依照規格產生程式，貼到下面對應的格子。
3. 執行**測試清單**，看懂每一句的 AI 原始回覆與系統解讀。
4. 照最後一格的**交件前檢查清單**交件。

**系統需求（簡化版）：** 科別只有內科、外科、牙科，每科每天有固定名額。功能有掛號、取消、查詢剩餘名額。
🔴 **安全規定：系統不可以提供任何醫療建議或根據症狀推薦科別。** 使用者沒說出科別時，請他直接說要掛哪一科，或洽詢櫃台。

# 📋 規格書（請直接在這一格修改）

學號：＿＿＿＿＿＿ 姓名：＿＿＿＿＿＿

## 1. 系統目的（一句話）
讓民眾能透過自然語言對話快速完成診所各科別的掛號、取消掛號與剩餘名額查詢，同時嚴格遵守不提供醫療建議之安全規範。

## 2. 使用者會怎麼說（至少 5 句）
1. 我想掛號看牙科
2. 幫我看一下內科今天還有沒有名額？
3. 我要退掛外科
4. 牙科還有位子嗎？沒有的話就算了
5. 我今天肚子很痛，請問要看哪一科？

## 3. 資料表（最多 2 張）

### 表一：科別名額表 (`departments`)
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| departments | dept_name | TEXT | 科別名稱（主鍵：內科、外科、牙科） |
| departments | total_quota | INTEGER | 當日總名額 |
| departments | remaining_quota | INTEGER | 當日剩餘名額 |

### 表二：掛號記錄表 (`appointments`)
| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| appointments | appt_id | INTEGER | 掛號編號（自動遞增） |
| appointments | user_id | TEXT | 使用者識別碼（預設或模擬用戶 ID） |
| appointments | dept_name | TEXT | 掛號科別 |
| appointments | status | TEXT | 狀態（ACTIVE: 已掛號, CANCELLED: 已取消） |

初始資料：
- `departments`：
  - 內科：總名額 3，剩餘名額 3
  - 外科：總名額 2，剩餘名額 2
  - 牙科：總名額 2，剩餘名額 2
- `appointments`：初始為空。

## 4. AI 的工作
- 輸入：使用者的一句話
- 輸出格式：`動作,對象`（例：`掛號,內科`）
- 動作只能是：`掛號`、`取消`、`查詢`、`未知`
- 對象只能是：`內科`、`外科`、`牙科`、`全部`、`未知`
- 無法判斷時輸出：`未知,未知`
- AI 不可以做的事：
  1. 嚴禁根據使用者描述的任何症狀推薦或推測科別。
  2. 嚴禁提供任何用藥、處置或醫療建議。
  3. 當使用者提及症狀、未指定具體科別，或科別不在「內科、外科、牙科」範圍內時，一律輸出 `未知,未知`，交由系統引導其說明科別或洽詢櫃台。

## 5. 功能與規則
| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 查詢餘額 | 對象為特定科別 | 顯示該科別目前剩餘名額（如：「內科目前剩餘名額：2」） | 無變化 |
| 查詢餘額 | 對象為全部或未指明 | 顯示三科當前所有剩餘名額 | 無變化 |
| 掛號成功 | 該科別 `remaining_quota > 0` 且用戶無該科有效掛號 | 「掛號成功！您已預約【科別】，號碼為【編號】。」 | 該科 `remaining_quota - 1`，新增記錄至 `appointments`（status='ACTIVE'） |
| 掛號失敗（額滿） | 該科別 `remaining_quota == 0` | 「抱歉，今日【科別】名額已滿，無法為您掛號。」 | 無變化 |
| 掛號失敗（重複） | 該用戶已有該科有效掛號 | 「您已掛號過【科別】，請勿重複掛號。」 | 無變化 |
| 取消成功 | 該用戶有該科狀態為 ACTIVE 之掛號 | 「已為您取消【科別】的掛號。」 | 該科 `remaining_quota + 1`，更新記錄狀態為 'CANCELLED' |
| 取消失敗 | 該用戶無該科有效掛號 | 「查無您的【科別】掛號記錄，無法取消。」 | 無變化 |
| 安全阻絕／引導 | AI 解析為 `未知,未知` 或未指定科別 | 「診所系統無法依據症狀推薦科別或提供醫療建議。請直接告知您欲掛號之科別（內科、外科、牙科），或請洽詢現場櫃台人員。」 | 無變化 |

## 6. 例外情況（至少 3 個）
1. 症狀詢問／醫療建議防護：使用者輸入「我頭暈發燒要看哪一科」或「吃止痛藥有效嗎」，系統不可推斷科別，必須安全婉拒並引導其指定科別或臨櫃。
2. 非支援科別處理：使用者輸入「我想掛眼科」或「小兒科還有名額嗎」，系統辨識科別非三科之一，輸出 `未知,未知` 並提示僅提供內科、外科、牙科。
3. 取消不存在的預約：使用者未曾掛號卻要求「取消牙科掛號」，系統檢查資料庫後明確回應查無紀錄，不誤退名額。

## 7. 測試清單（至少 8 句，正常／錯誤／邊界都要有）
| # | 類型 | 使用者說的話 | 預期結果 |
|---|---|---|---|
| 1 | 正常 | 請問牙科今天還有名額嗎？ | `查詢,牙科` ➔ 顯示牙科剩餘名額 |
| 2 | 正常 | 我想要掛號內科 | `掛號,內科` ➔ 掛號成功，內科名額減 1 |
| 3 | 正常 | 幫我取消剛剛掛的內科 | `取消,內科` ➔ 取消成功，內科名額加 1 |
| 4 | 安全規定 | 我肚子很痛拉肚子，要看哪一科？ | `未知,未知` ➔ 觸發安全提示，拒絕推薦科別並請其指定科別或洽櫃台 |
| 5 | 錯誤 | 請幫我掛眼科門診 | `未知,未知` ➔ 提示無法辨識該科別，僅支援內科/外科/牙科 |
| 6 | 錯誤 | 我要取消外科掛號 | `取消,外科` ➔ 系統查無掛號記錄，回應無法取消 |
| 7 | 邊界 | 我要掛內科（名額已為 0 時執行） | `掛號,內科` ➔ 系統判定名額為 0，回應額滿無法掛號 |
| 8 | 邊界 | 我要掛牙科（已有該科有效掛號時執行） | `掛號,牙科` ➔ 系統判定已有有效掛號，回應請勿重複掛號 |

## 🤖 可以使用的 prompt

**① 請 AI 寫規格書初稿**
```
我要做一個「診所掛號系統」的課堂小作品（Google Colab + SQLite + Groq + Gradio）。
請依照下面的規格範本，幫我寫出一份規格書初稿。要求：
1. 科別只有：內科、外科、牙科，每科每天有固定名額。
2. 功能：掛號、取消掛號、查詢某科剩餘名額。
3. 資料表不超過 2 張。
4. AI 只負責判斷「動作」與「科別」。如果使用者只描述症狀、沒有說出科別，AI 一律回答未知；系統絕對不可以提供任何醫療建議或推薦科別。
5. 測試清單至少 8 句，要包含正常、錯誤、邊界三種情況，並寫出預期結果。
規格範本如下：
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫一份 Google Colab 程式，並遵守以下要求：
1. 第 1 格（安裝套件與讀取金鑰）我已經有了，請從第 2 格開始寫，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) 測試清單 (6) 功能函式 (7) 主流程與完整流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經定義好 client（Groq）與 MODEL，也已經 import 了 sqlite3、pandas as pd、display、gradio as gr，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 temperature=0，並用 try/except 包起來，失敗時回傳「未知」，不可讓程式當掉。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」。
7. fetchone() 之後一定要先檢查是否為 None。
8. 資料庫連線用 try/finally 確保關閉。
9. 測試清單的格式為 (使用者說的話, 預期動作, 預期科別)，跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 只用 sqlite3、groq、gradio、pandas，不要用其他套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

In [34]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [35]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# 提醒：每次執行都要 DROP TABLE IF EXISTS 再重建
# ============================================================

# 定義資料庫檔案名稱
DB_NAME = "clinic.db"

# 建立資料庫連線，並使用 try/finally 確保一定會關閉連線
conn = sqlite3.connect(DB_NAME)
try:
    cursor = conn.cursor()

    # 1. 每次執行先刪除舊資料表（確保測試時每次都是初始狀態）
    cursor.execute("DROP TABLE IF EXISTS appointments")
    cursor.execute("DROP TABLE IF EXISTS departments")

    # 2. 建立科別名額表 (departments)
    cursor.execute("""
        CREATE TABLE departments (
            dept_name TEXT PRIMARY KEY,
            total_quota INTEGER NOT NULL,
            remaining_quota INTEGER NOT NULL
        )
    """)

    # 3. 建立掛號記錄表 (appointments)
    cursor.execute("""
        CREATE TABLE appointments (
            appt_id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT NOT NULL,
            dept_name TEXT NOT NULL,
            status TEXT NOT NULL,
            FOREIGN KEY (dept_name) REFERENCES departments(dept_name)
        )
    """)

    # 4. 插入初始資料（使用 ? 參數化，避免 SQL Injection）
    # 內科 3 位、外科 2 位、牙科 2 位
    initial_departments = [
        ("內科", 3, 3),
        ("外科", 2, 2),
        ("牙科", 2, 2)
    ]
    cursor.executemany(
        "INSERT INTO departments (dept_name, total_quota, remaining_quota) VALUES (?, ?, ?)",
        initial_departments
    )

    # 提交資料庫變更
    conn.commit()
    print("✅ 資料庫初始化成功！已建立 departments 與 appointments 表。")

finally:
    # 確保關閉資料庫連線，不佔用資源
    conn.close()

✅ 資料庫初始化成功！已建立 departments 與 appointments 表。


In [36]:
# ============================================================
# 第 3 格：顯示資料表
# 提醒：定義 show_tables() 並呼叫一次
# ============================================================

def show_tables():
    """查詢並以表格形式顯示 departments 與 appointments 兩個資料表的現況"""
    conn = sqlite3.connect(DB_NAME)
    try:
        print("📌 【科別名額表 departments】")
        # 使用 pandas 的 read_sql_query 將查詢結果轉為美觀的 DataFrame
        df_depts = pd.read_sql_query("SELECT * FROM departments", conn)
        display(df_depts)

        print("\n📌 【掛號記錄表 appointments】")
        df_appts = pd.read_sql_query("SELECT * FROM appointments", conn)
        if df_appts.empty:
            print("（目前尚無掛號記錄）")
        else:
            display(df_appts)
    finally:
        # 確保關閉連線
        conn.close()

# 呼叫一次以確認初始資料正確寫入
show_tables()

📌 【科別名額表 departments】


,dept_name,total_quota,remaining_quota
0,內科,3,3
1,外科,2,2
2,牙科,2,2



📌 【掛號記錄表 appointments】
（目前尚無掛號記錄）


In [37]:
# ============================================================
# 第 4 格：AI 解讀函式（徹底解決空白回覆與誤判問題）
# ============================================================

def interpret_intent(user_text):
    """
    使用 Groq 模型解析使用者意圖。
    輸出格式必須為「動作,對象」，嚴禁推論或提供醫療建議。
    回傳值：(動作, 科別, 原始回覆)
    """
    VALID_ACTIONS = ["掛號", "取消", "查詢"]

    ACTION_MAPPING = {
        "掛號": "掛號", "預約": "掛號",
        "取消": "取消", "退掛": "取消",
        "查詢": "查詢"
    }

    DEPT_MAPPING = {
        "內科": "內科", "內": "內科",
        "外科": "外科", "外": "外科",
        "牙科": "牙科", "牙": "牙科"
    }

    system_prompt = (
        "你是一個診所掛號系統的意圖辨識助手。\n"
        "請根據使用者的語句，直接輸出「動作,對象」，不要輸出任何其他字元、空格或思考過程。\n"
        "【動作只能是】：掛號、取消、查詢、未知\n"
        "【對象只能是】：內科、外科、牙科、未知\n"
        "\n"
        "🔴 核心規則（優先權最高）：\n"
        "1. 嚴禁任何醫療建議！若使用者描述症狀（如頭痛、發燒、肚子痛、拉肚子等）、問該看哪科或詢問處置，一律只輸出：未知,未知\n"
        "2. 若使用者說要預約/掛號未支援的科別（如眼科、耳鼻喉科、小兒科），輸出：掛號,未知\n"
        "3. 「預約」等同於「掛號」；「退掛」等同於「取消」。\n"
        "4. 無法確定時，一律輸出：未知,未知\n"
        "\n"
        "範例輸出：\n"
        "- 「請問今天牙科還有名額嗎？」 -> 查詢,牙科\n"
        "- 「我要掛號內科」 -> 掛號,內科\n"
        "- 「幫我取消剛才的內科掛號」 -> 取消,內科\n"
        "- 「我頭痛發燒好幾天了，要掛哪一科？」 -> 未知,未知\n"
        "- 「肚子劇痛可以幫我掛號嗎？」 -> 未知,未知\n"
        "- 「請幫我預約眼科」 -> 掛號,未知\n"
        "- 「我要取消外科」 -> 取消,外科"
    )

    raw_response = "未知"

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            model=MODEL,
            temperature=0,
            max_tokens=150
        )
        raw_response = chat_completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ 呼叫 Groq API 發生例外：{e}")
        return "未知", "未知", "API 呼叫失敗"

    parsed_action = "未知"
    parsed_dept = "未知"

    # 1. 優先依逗號格式解析
    if "," in raw_response:
        parts = [p.strip() for p in raw_response.split(",", 1)]
        action_candidate = parts[0]
        dept_candidate = parts[1]

        if action_candidate in VALID_ACTIONS:
            parsed_action = action_candidate
        elif action_candidate in ACTION_MAPPING:
            parsed_action = ACTION_MAPPING[action_candidate]

        if dept_candidate in DEPT_MAPPING:
            parsed_dept = DEPT_MAPPING[dept_candidate]

    # 2. 備用容錯：僅從 AI 的 raw_response 尋找關鍵字（絕對不搜尋 user_text）
    else:
        for key, val in ACTION_MAPPING.items():
            if key in raw_response:
                parsed_action = val
                break
        for key, val in DEPT_MAPPING.items():
            if key in raw_response:
                parsed_dept = val
                break

    # 3. 症狀安全保險：若使用者純粹問症狀但 AI 吐出空值，預設為 未知,未知
    if raw_response == "" or raw_response == "未知":
        parsed_action = "未知"
        parsed_dept = "未知"

    return parsed_action, parsed_dept, raw_response

In [31]:
# ============================================================
# 第 5 格：測試清單
# 提醒：至少 8 句，要與規格書第 7 節一致
# ============================================================

# 測試清單資料結構：(使用者說的話, 預期動作, 預期科別)
# 涵蓋正常操作、安全規範（問診/症狀）、錯誤輸入與邊界情況
test_cases = [
    ("請問今天牙科還有名額嗎？", "查詢", "牙科"),
    ("我要掛號內科", "掛號", "內科"),
    ("幫我取消剛才的內科掛號", "取消", "內科"),
    ("我頭痛發燒好幾天了，要掛哪一科？", "未知", "未知"),
    ("肚子劇痛可以幫我掛號嗎？", "未知", "未知"),
    ("請幫我預約眼科", "掛號", "未知"),
    ("我要取消外科", "取消", "外科"),
    ("我要掛號牙科", "掛號", "牙科"),
    ("我想掛內科", "掛號", "內科")
]

print(f"📋 測試清單載入完成，共 {len(test_cases)} 筆測試語句。")
print("-" * 70)
print(f"{'#':<3} | {'預期動作':<6} | {'預期科別':<6} | {'測試語句'}")
print("-" * 70)
for idx, (text, exp_action, exp_dept) in enumerate(test_cases, start=1):
    print(f"{idx:<3} | {exp_action:<6} | {exp_dept:<6} | {text}")
print("-" * 70)

# 執行測試清單並驗證 AI 解讀函式
passed_count = 0
total_count = len(test_cases)

print("\n🚀 開始執行 AI 解讀測試：\n")

for idx, (text, exp_act, exp_dept) in enumerate(test_cases, start=1):
    act, dept, raw = interpret_intent(text)
    is_passed = (act == exp_act) and (dept == exp_dept)

    if is_passed:
        passed_count += 1
        status = "✅ 通過"
    else:
        status = "❌ 失敗"

    print(f"[{idx}/{total_count}] {status}")
    print(f"  🗣️ 使用者輸入：{text}")
    print(f"  🤖 AI 原始回覆：{raw}")
    print(f"  ⚙️ 系統解讀：動作={act}, 科別={dept} (預期: 動作={exp_act}, 科別={exp_dept})")
    print("-" * 50)

print(f"\n📊 測試結果總計：通過 {passed_count} / {total_count} 句（通過率：{passed_count / total_count * 100:.1f}%）")

📋 測試清單載入完成，共 9 筆測試語句。
----------------------------------------------------------------------
#   | 預期動作   | 預期科別   | 測試語句
----------------------------------------------------------------------
1   | 查詢     | 牙科     | 請問今天牙科還有名額嗎？
2   | 掛號     | 內科     | 我要掛號內科
3   | 取消     | 內科     | 幫我取消剛才的內科掛號
4   | 未知     | 未知     | 我頭痛發燒好幾天了，要掛哪一科？
5   | 未知     | 未知     | 肚子劇痛可以幫我掛號嗎？
6   | 掛號     | 未知     | 請幫我預約眼科
7   | 取消     | 外科     | 我要取消外科
8   | 掛號     | 牙科     | 我要掛號牙科
9   | 掛號     | 內科     | 我想掛內科
----------------------------------------------------------------------

🚀 開始執行 AI 解讀測試：

[1/9] ✅ 通過
  🗣️ 使用者輸入：請問今天牙科還有名額嗎？
  🤖 AI 原始回覆：查詢,牙科
  ⚙️ 系統解讀：動作=查詢, 科別=牙科 (預期: 動作=查詢, 科別=牙科)
--------------------------------------------------
[2/9] ✅ 通過
  🗣️ 使用者輸入：我要掛號內科
  🤖 AI 原始回覆：掛號,內科
  ⚙️ 系統解讀：動作=掛號, 科別=內科 (預期: 動作=掛號, 科別=內科)
--------------------------------------------------
[3/9] ✅ 通過
  🗣️ 使用者輸入：幫我取消剛才的內科掛號
  🤖 AI 原始回覆：取消,內科
  ⚙️ 系統解讀：動作=取消, 科別=內科 (預期: 動作=取消, 科別=內科)
--------------------------------

In [38]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# 提醒：fetchone() 後先檢查 None；用 try/finally 關閉連線
# ============================================================

def query_quota(dept_name):
    """
    查詢指定科別的剩餘名額
    回傳值：系統回應字串
    """
    conn = sqlite3.connect(DB_NAME)
    try:
        cursor = conn.cursor()
        # 使用 ? 參數化查詢科別資訊
        cursor.execute(
            "SELECT total_quota, remaining_quota FROM departments WHERE dept_name = ?",
            (dept_name,)
        )
        row = cursor.fetchone()

        # 嚴格檢查查詢結果是否為 None
        if row is None:
            return f"❌ 系統查無【{dept_name}】的資料，請確認科別是否正確。"

        total_quota, remaining_quota = row
        return f"ℹ️ 【{dept_name}】目前剩餘名額：{remaining_quota} 位（當日總名額：{total_quota} 位）。"

    finally:
        # 確保關閉資料庫連線
        conn.close()


def register_appointment(user_id, dept_name):
    """
    處理掛號邏輯：
    1. 檢查科別是否存在
    2. 檢查該用戶是否已有該科的有效預約（ACTIVE），避免重複掛號
    3. 檢查名額是否大於 0
    4. 扣減名額並寫入掛號記錄
    回傳值：系統回應字串
    """
    conn = sqlite3.connect(DB_NAME)
    try:
        cursor = conn.cursor()

        # 1. 檢查科別是否存在及目前剩餘名額
        cursor.execute(
            "SELECT remaining_quota FROM departments WHERE dept_name = ?",
            (dept_name,)
        )
        dept_row = cursor.fetchone()
        if dept_row is None:
            return f"❌ 查無【{dept_name}】資訊，目前僅支援內科、外科、牙科。"

        remaining_quota = dept_row[0]

        # 2. 檢查用戶是否已有該科的有效掛號 (ACTIVE)
        cursor.execute(
            "SELECT appt_id FROM appointments WHERE user_id = ? AND dept_name = ? AND status = 'ACTIVE'",
            (user_id, dept_name)
        )
        existing_appt = cursor.fetchone()
        if existing_appt is not None:
            return f"⚠️ 您已有【{dept_name}】的預約紀錄（掛號編號：#{existing_appt[0]}），請勿重複掛號。"

        # 3. 檢查名額是否已額滿
        if remaining_quota <= 0:
            return f"🚫 抱歉，今日【{dept_name}】名額已滿，無法辦理掛號。"

        # 4. 執行掛號：扣減名額並新增記錄
        cursor.execute(
            "UPDATE departments SET remaining_quota = remaining_quota - 1 WHERE dept_name = ?",
            (dept_name,)
        )
        cursor.execute(
            "INSERT INTO appointments (user_id, dept_name, status) VALUES (?, ?, 'ACTIVE')",
            (user_id, dept_name)
        )
        appt_id = cursor.lastrowid

        # 提交事務
        conn.commit()
        return f"🎉 掛號成功！已為您預約【{dept_name}】，您的掛號號碼為：#{appt_id}。"

    finally:
        # 確保關閉資料庫連線
        conn.close()


def cancel_appointment(user_id, dept_name):
    """
    處理取消掛號邏輯：
    1. 檢查該用戶是否有該科有效預約（ACTIVE）
    2. 更新記錄狀態為 CANCELLED
    3. 釋出名額（remaining_quota + 1）
    回傳值：系統回應字串
    """
    conn = sqlite3.connect(DB_NAME)
    try:
        cursor = conn.cursor()

        # 1. 查詢用戶是否有該科的有效掛號
        cursor.execute(
            "SELECT appt_id FROM appointments WHERE user_id = ? AND dept_name = ? AND status = 'ACTIVE' ORDER BY appt_id DESC LIMIT 1",
            (user_id, dept_name)
        )
        appt_row = cursor.fetchone()

        # 嚴格檢查是否有找到有效預約
        if appt_row is None:
            return f"⚠️ 查無您的【{dept_name}】有效掛號記錄，無法辦理取消。"

        target_appt_id = appt_row[0]

        # 2. 更新記錄狀態為 CANCELLED
        cursor.execute(
            "UPDATE appointments SET status = 'CANCELLED' WHERE appt_id = ?",
            (target_appt_id,)
        )

        # 3. 將該科剩餘名額加回 1
        cursor.execute(
            "UPDATE departments SET remaining_quota = remaining_quota + 1 WHERE dept_name = ?",
            (dept_name,)
        )

        # 提交事務
        conn.commit()
        return f"✅ 已成功為您取消【{dept_name}】的掛號（原號碼：#{target_appt_id}）。"

    finally:
        # 確保關閉資料庫連線
        conn.close()

print("✅ 功能函式（query_quota, register_appointment, cancel_appointment）定義完成！")

✅ 功能函式（query_quota, register_appointment, cancel_appointment）定義完成！
✅ 功能函式（query_quota, register_appointment, cancel_appointment）定義完成！


In [39]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# 提醒：最後呼叫 show_tables() 確認資料庫有變化
# ============================================================

def process_request(user_text, user_id="U001"):
    """
    主流程函式：
    1. 呼叫 interpret_intent 解析意圖 (動作, 科別, 原始回覆)
    2. 判斷安全規則：若動作或科別包含「未知」，直接提供安全引導，拒絕醫療諮詢
    3. 根據合法動作 (查詢、掛號、取消) 呼叫對應功能函式
    回傳值：(系統回應文字, 動作, 科別, 原始回覆)
    """
    action, dept, raw_response = interpret_intent(user_text)

    # 安全防護與格式防禦：未指定科別、詢問症狀或超出科別範圍
    if action == "未知" or dept == "未知":
        reply = (
            "⚠️ 本系統為自動掛號工具，無法根據症狀評估病況或推薦科別。\n"
            "請直接告知欲辦理之科別（目前僅提供：內科、外科、牙科），或請洽詢現場櫃台人員。"
        )
        return reply, action, dept, raw_response

    # 依照動作派發到對應功能
    if action == "查詢":
        reply = query_quota(dept)
    elif action == "掛號":
        reply = register_appointment(user_id, dept)
    elif action == "取消":
        reply = cancel_appointment(user_id, dept)
    else:
        reply = "⚠️ 無法辨識您的具體需求，請重新輸入。"

    return reply, action, dept, raw_response


# ============================================================
# 完整流程自動化測試（模擬使用者實際互動情境）
# ============================================================
print("🧪 開始執行主流程與業務邏輯測試...\n")

# 定義多輪情境測試案例
flow_tests = [
    ("測試 1：正常查詢", "請問今天牙科還有名額嗎？"),
    ("測試 2：正常掛號", "我要掛號內科"),
    ("測試 3：重複掛號阻絕（邊界）", "我要掛號內科"),
    ("測試 4：純症狀詢問（安全防護）", "我最近肚子一直絞痛還拉肚子，要看哪一科？"),
    ("測試 5：不支援科別（錯誤輸入）", "請幫我預約眼科"),
    ("測試 6：正常取消掛號", "幫我取消剛才的內科掛號"),
    ("測試 7：無紀錄取消（錯誤處理）", "我要取消外科掛號"),
    ("測試 8：掛號牙科", "我想掛牙科門診"),
]

for title, user_input in flow_tests:
    print(f"▶️ 【{title}】")
    print(f"  🗣️ 使用者輸入：{user_input}")
    reply, action, dept, raw = process_request(user_input, user_id="U001")
    print(f"  🤖 AI 解析：動作={action}, 科別={dept} (原始: '{raw}')")
    print(f"  💬 系統回應：{reply}")
    print("-" * 65)

# 測試結束後呼叫 show_tables() 檢查資料庫變化
print("\n📊 測試完成！檢視資料庫最終狀態：")
show_tables()

🧪 開始執行主流程與業務邏輯測試...

▶️ 【測試 1：正常查詢】
  🗣️ 使用者輸入：請問今天牙科還有名額嗎？
  🤖 AI 解析：動作=查詢, 科別=牙科 (原始: '查詢,牙科')
  💬 系統回應：ℹ️ 【牙科】目前剩餘名額：2 位（當日總名額：2 位）。
-----------------------------------------------------------------
▶️ 【測試 2：正常掛號】
  🗣️ 使用者輸入：我要掛號內科
  🤖 AI 解析：動作=掛號, 科別=內科 (原始: '掛號,內科')
  💬 系統回應：🎉 掛號成功！已為您預約【內科】，您的掛號號碼為：#1。
-----------------------------------------------------------------
▶️ 【測試 3：重複掛號阻絕（邊界）】
  🗣️ 使用者輸入：我要掛號內科
  🤖 AI 解析：動作=掛號, 科別=內科 (原始: '掛號,內科')
  💬 系統回應：⚠️ 您已有【內科】的預約紀錄（掛號編號：#1），請勿重複掛號。
-----------------------------------------------------------------
▶️ 【測試 4：純症狀詢問（安全防護）】
  🗣️ 使用者輸入：我最近肚子一直絞痛還拉肚子，要看哪一科？
  🤖 AI 解析：動作=未知, 科別=未知 (原始: '未知,未知')
  💬 系統回應：⚠️ 本系統為自動掛號工具，無法根據症狀評估病況或推薦科別。
請直接告知欲辦理之科別（目前僅提供：內科、外科、牙科），或請洽詢現場櫃台人員。
-----------------------------------------------------------------
▶️ 【測試 5：不支援科別（錯誤輸入）】
  🗣️ 使用者輸入：請幫我預約眼科
  🤖 AI 解析：動作=掛號, 科別=未知 (原始: '掛號,未知')
  💬 系統回應：⚠️ 本系統為自動掛號工具，無法根據症狀評估病況或推薦科別。
請直接告知欲辦理之科別（目前僅提供：內科、外科、牙科），或請洽詢現場櫃台人員。
---------------------------------------------

,dept_name,total_quota,remaining_quota
0,內科,3,3
1,外科,2,2
2,牙科,2,1



📌 【掛號記錄表 appointments】


,appt_id,user_id,dept_name,status
0,1,U001,內科,CANCELLED
1,2,U001,牙科,ACTIVE


🧪 開始執行主流程與業務邏輯測試...

▶️ 【測試 1：正常查詢】
  🗣️ 使用者輸入：請問今天牙科還有名額嗎？
  🤖 AI 解析：動作=查詢, 科別=牙科 (原始: '查詢,牙科')
  💬 系統回應：ℹ️ 【牙科】目前剩餘名額：2 位（當日總名額：2 位）。
-----------------------------------------------------------------
▶️ 【測試 2：正常掛號】
  🗣️ 使用者輸入：我要掛號內科
  🤖 AI 解析：動作=掛號, 科別=內科 (原始: '掛號,內科')
  💬 系統回應：🎉 掛號成功！已為您預約【內科】，您的掛號號碼為：#1。
-----------------------------------------------------------------
▶️ 【測試 3：重複掛號阻絕（邊界）】
  🗣️ 使用者輸入：我要掛號內科
  🤖 AI 解析：動作=掛號, 科別=內科 (原始: '掛號,內科')
  💬 系統回應：⚠️ 您已有【內科】的預約紀錄（掛號編號：#1），請勿重複掛號。
-----------------------------------------------------------------
▶️ 【測試 4：純症狀詢問（安全防護）】
  🗣️ 使用者輸入：我最近肚子一直絞痛還拉肚子，要看哪一科？
  🤖 AI 解析：動作=未知, 科別=未知 (原始: '未知,未知')
  💬 系統回應：⚠️ 本系統為自動掛號工具，無法根據症狀評估病況或推薦科別。
請直接告知欲辦理之科別（目前僅提供：內科、外科、牙科），或請洽詢現場櫃台人員。
-----------------------------------------------------------------
▶️ 【測試 5：不支援科別（錯誤輸入）】
  🗣️ 使用者輸入：請幫我預約眼科
  🤖 AI 解析：動作=掛號, 科別=未知 (原始: '掛號,未知')
  💬 系統回應：⚠️ 本系統為自動掛號工具，無法根據症狀評估病況或推薦科別。
請直接告知欲辦理之科別（目前僅提供：內科、外科、牙科），或請洽詢現場櫃台人員。
---------------------------------------------

,dept_name,total_quota,remaining_quota
0,內科,3,3
1,外科,2,2
2,牙科,2,1



📌 【掛號記錄表 appointments】


,appt_id,user_id,dept_name,status
0,1,U001,內科,CANCELLED
1,2,U001,牙科,ACTIVE


In [42]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：放在最後一格；GitHub 上看不到網頁，批改看的是第 5、7 格的結果
# ============================================================

def get_status_dataframes():
    """輔助函式：讀取最新的資料表內容供 Gradio 介面即時顯示"""
    conn = sqlite3.connect(DB_NAME)
    try:
        df_depts = pd.read_sql_query("SELECT * FROM departments", conn)
        df_appts = pd.read_sql_query("SELECT * FROM appointments", conn)
        return df_depts, df_appts
    finally:
        conn.close()

def clinic_web_interface(user_message, user_id):
    """
    Gradio 互動函式：
    接收使用者訊息與用戶 ID，呼叫第 7 格的主流程 process_request，
    並同步回傳系統回應、AI 解讀細節以及最新資料庫狀態。
    """
    if not user_message or not user_message.strip():
        df_depts, df_appts = get_status_dataframes()
        return "請輸入想諮詢或辦理的內容。", "", "", "", df_depts, df_appts

    # 預設使用者 ID（若使用者未輸入則使用 U001）
    actual_user_id = user_id.strip() if user_id and user_id.strip() else "U001"

    # 呼叫主流程
    reply, action, dept, raw_resp = process_request(user_message, user_id=actual_user_id)

    # 取得最新資料庫狀態
    df_depts, df_appts = get_status_dataframes()

    return reply, action, dept, raw_resp, df_depts, df_appts


# 建立 Gradio Blocks 介面
with gr.Blocks(title="診所掛號智慧助理") as demo:
    gr.Markdown("## 🏥 診所自然語言掛號系統")
    gr.Markdown(
        "支援功能：**掛號**、**取消掛號**、**查詢剩餘名額**。<br>"
        "支援科別：**內科**、**外科**、**牙科**。<br>"
        "🔴 *安全聲明：本系統不提供任何醫療診斷或症狀建議，如不適請依指引掛號或洽詢現場櫃台。*"
    )

    with gr.Row():
        with gr.Column(scale=1):
            user_id_input = gr.Textbox(value="U001", label="病患/使用者編號 (User ID)")
            msg_input = gr.Textbox(
                placeholder="例如：請問今天牙科還有名額嗎？ / 我要掛號內科",
                label="請輸入您的需求或問題",
                lines=2
            )
            submit_btn = gr.Button("發送訊息", variant="primary")

            # 提供快速點擊測試的範例句子
            gr.Examples(
                examples=[
                    ["請問今天牙科還有名額嗎？", "U001"],
                    ["我要掛號內科", "U001"],
                    ["幫我取消剛才的內科掛號", "U001"],
                    ["我頭痛發燒好幾天了，要掛哪一科？", "U001"],
                    ["請幫我預約眼科", "U001"],
                    ["我要取消外科", "U001"]
                ],
                inputs=[msg_input, user_id_input],
                label="💡 常用測試範例（點擊直接帶入）"
            )

        with gr.Column(scale=1):
            reply_output = gr.Textbox(label="💬 系統回應", lines=4)
            with gr.Row():
                action_output = gr.Textbox(label="判斷動作")
                dept_output = gr.Textbox(label="判斷科別")
            raw_output = gr.Textbox(label="🤖 AI 原始輸出 (Raw Output)")

    gr.Markdown("### 📊 即時資料庫狀態 (Live Database)")
    with gr.Row():
        depts_table = gr.Dataframe(label="科別名額表 (departments)")
        appts_table = gr.Dataframe(label="掛號記錄表 (appointments)")

    # 綁定按鈕與 Enter 送出事件
    submit_btn.click(
        fn=clinic_web_interface,
        inputs=[msg_input, user_id_input],
        outputs=[reply_output, action_output, dept_output, raw_output, depts_table, appts_table]
    )
    msg_input.submit(
        fn=clinic_web_interface,
        inputs=[msg_input, user_id_input],
        outputs=[reply_output, action_output, dept_output, raw_output, depts_table, appts_table]
    )

    # 頁面載入時先初始化顯示資料表內容
    demo.load(fn=get_status_dataframes, outputs=[depts_table, appts_table])

# 啟動 Gradio 服務
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://285d79646dd5ea4194.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://285d79646dd5ea4194.gradio.live


# ✅ 交件前檢查清單（每一項都要打勾）

- [ ] 程式裡**沒有**任何 `gsk_` 開頭的金鑰，也沒有 print 出金鑰
- [ ] 規格書第 1～7 節都填完，測試清單至少 8 句
- [ ] 選單「執行階段 → 重新啟動並執行全部」，**全部跑完沒有紅色錯誤**
- [ ] 每一格左邊的執行編號是**連續遞增**的（1、2、3…）
- [ ] 選單「編輯 → 筆記本設定」，**沒有**勾選「儲存筆記本時忽略程式碼儲存格輸出內容」
- [ ] 選單「檔案 → 在 GitHub 中儲存副本」，存到**公開（Public）**的 repo
- [ ] 用無痕視窗打開 GitHub 上的檔案，確認看得到程式和執行結果